In [6]:
# Python bibliotek
import tkinter as tk
from pathlib import Path
from tkinter import ttk, messagebox

# Lokala programfiler
from src.utils import *
from src.inventory import InventoryManager
from src.database import DatabaseManager
from src.recipes import RecipeManager
from src.shopping import ShoppingManager
from src.models import Ingredient, Recipe, ShoppingList
from src.sql_parser import SQLiteParser



In [27]:
# Pickle-baserade databasfiler
using_sql = False

if not using_sql:
    inv_db = DatabaseManager(Path("data/inventory.pkl"), default_factory=dict)
    rec_db = DatabaseManager(Path("data/recipes.pkl"), default_factory=dict)
    shop_db = DatabaseManager(Path("data/shopping.pkl"), default_factory=dict)
else:
    # Eventuell SQLite implementering
    pass

# Kopplar databaser till manager-objekter.
inventory = InventoryManager(inv_db)
recipes = RecipeManager(rec_db)
shopping = ShoppingManager(shop_db)

# SQL-parser för demonstration
sql_parser = SQLiteParser("data/sql_demo.db")


In [28]:


class App:
    def __init__(self, root, inventory_manager, recipe_manager, shopping_manager, sql_parser):

        self.root = root
        self.inventory = inventory_manager
        self.recipes = recipe_manager
        self.shopping = shopping_manager
        self.sql_parser = sql_parser

        self.root.title("Matplanerare – Pickle & SQL Jämförelse")
        self.root.geometry("950x600")

        self.allowed_units = ["st", "g", "kg", "ml", "l", "tsk", "msk", "dl"]

        # Skapar notebook med flikar
        notebook = ttk.Notebook(root)
        notebook.pack(fill="both", expand=True)

        # Skapar interface flikar
        self.tab_inventory = ttk.Frame(notebook)
        self.tab_recipes = ttk.Frame(notebook)
        self.tab_shopping = ttk.Frame(notebook)
        self.tab_sql = ttk.Frame(notebook)

        # Lägger till flikar i notebook
        notebook.add(self.tab_inventory, text="Inventarie")
        notebook.add(self.tab_recipes, text="Recept")
        notebook.add(self.tab_shopping, text="Inköpslistor")
        notebook.add(self.tab_sql, text="SQL-jämförelse")

        # Bygger interface
        self.build_inventory_tab()
        self.build_recipes_tab()
        self.build_shopping_tab()
        self.build_sql_tab()

    # Inventory flik
    def build_inventory_tab(self) -> None:
        """
        Bygger fönstret för inventory-fliken
        """

        container = self.tab_inventory # Aktuell flik

        form_frame = ttk.LabelFrame(container, text="Lägg till / uppdatera ingrediens")
        form_frame.pack(fill="x", padx=10, pady=5)

        # Skapar rubriker
        ttk.Label(form_frame, text="Namn:").grid(row=0, column=0)
        ttk.Label(form_frame, text="Mängd:").grid(row=0, column=2)
        ttk.Label(form_frame, text="Enhet:").grid(row=0, column=4)

        # Initierar attribut för popup
        self.inventory_name_var = tk.StringVar()
        self.inventory_quantity_var = tk.DoubleVar()
        self.inventory_unit_var = tk.StringVar()

        # Skapar entry-felt
        ttk.Entry(form_frame, textvariable=self.inventory_name_var, width=20).grid(row=0, column=1)
        ttk.Entry(form_frame, textvariable=self.inventory_quantity_var, width=10).grid(row=0, column=3)

        # Skapar drop-down-felt
        inventory_unit_combobox = ttk.Combobox(form_frame,textvariable=self.inventory_unit_var,values=self.allowed_units,width=10,state="readonly")
        inventory_unit_combobox.grid(row=0, column=5)
        inventory_unit_combobox.set("st")

        # Skapar knapp-felt
        ttk.Button(form_frame, text="Lägg till / uppdatera", command=self.add_inventory_item).grid(row=0, column=6, padx=5)
        ttk.Button(form_frame, text="Ta bort vald", command=self.remove_selected_inventory_item).grid(row=0, column=7)
        ttk.Button(form_frame, text="Spara", command=self.save_inventory_to_disk).grid(row=0, column=8, padx=5)

        # Skapar tabellruta
        table_frame = ttk.LabelFrame(container, text="Inventarieinnehåll")
        table_frame.pack(fill="both", expand=True, padx=10, pady=10)

        # Skapar layout
        self.inventory_tree = ttk.Treeview(table_frame, columns=("Namn", "Mängd", "Enhet"), show="headings")
        self.inventory_tree.heading("Namn", text="Namn")
        self.inventory_tree.heading("Mängd", text="Mängd")
        self.inventory_tree.heading("Enhet", text="Enhet")
        self.inventory_tree.pack(fill="both", expand=True)

        self.refresh_inventory_view() # Uppdaterar gränssnitt

    def refresh_inventory_view(self) -> None:
        """
        Uppdaterar gränssnittet för att reflektera ändringar i dess innehåll
        """
        for row_id in self.inventory_tree.get_children():
            # Tar bort rader med borttagna objekt
            self.inventory_tree.delete(row_id)

        for ingredient in self.inventory.list_items():
            # Lägger till nytt objekt i tabellen
            self.inventory_tree.insert("", "end", values=(ingredient.name, round(ingredient.quantity, 2), ingredient.unit))

    def add_inventory_item(self) -> None:
        """
        Lägger in tillagda ingredienser i gränssnittet
        """
        name = self.inventory_name_var.get().strip()
        quantity = self.inventory_quantity_var.get()
        unit = self.inventory_unit_var.get().strip()

        # Felhantering
        if not name or not unit:
            messagebox.showerror("Fel", "Du måste ange både namn och enhet.")
            return

        self.inventory.add(Ingredient(name, quantity, unit)) # Lägg till
        self.refresh_inventory_view()

    def remove_selected_inventory_item(self) -> None:
        """
        Tar bort ingrediens ur gränssnittet
        """
        selected_rows = self.inventory_tree.selection() # Hämta vald rad


        if not selected_rows:
            return

        # Hämta namn
        name = self.inventory_tree.item(selected_rows[0])["values"][0]

        # Bekräftelse fönster för användarvänlighet
        if messagebox.askyesno("Bekräfta", f"Vill du ta bort {name}?"):
            self.inventory.remove(name) # ta bort
            self.refresh_inventory_view()

    def save_inventory_to_disk(self) -> None: # Bör byggas in flytande i koden
        """
        Metod för att spara inventariet
        """
        self.inventory.save()
        messagebox.showinfo("Sparat", "Inventariet har sparats!")

    """ Receptflik """
    def build_recipes_tab(self) -> None:
        """
        Bygger fönstret för recept-fliken
        """
        container = self.tab_recipes # Koppla till flik

        # Skapa kommandofönster
        toolbar = ttk.Frame(container)
        toolbar.pack(fill="x", padx=10, pady=5)

        # Skapa knappar och fäst på fönster
        ttk.Button(toolbar, text="Lägg till recept", command=self.open_new_recipe_popup).pack(side="left", padx=5)
        ttk.Button(toolbar, text="Hantera recept", command=self.open_selected_recipe_ingredients_window).pack(side="left", padx=5)

        # Skapa attribut för sidan
        self.recipes_tree = ttk.Treeview(container, columns=("Recept", "Portioner"), show="headings")
        self.recipes_tree.heading("Recept", text="Recept")
        self.recipes_tree.heading("Portioner", text="Portioner")
        self.recipes_tree.pack(fill="both", expand=True, padx=10, pady=10)

        # Intuitiv mapping, öppnar recept fönster anonymt
        self.recipes_tree.bind("<Double-1>", lambda event: self.open_selected_recipe_ingredients_window())

        self.refresh_recipes_view()

    def refresh_recipes_view(self) -> None:
        """
        Uppdatera sidan för att visa förändringar
        """
        for row_id in self.recipes_tree.get_children():
            self.recipes_tree.delete(row_id) # Rensa borttagna objekt

        for recipe in self.recipes.list_recipes():
            self.recipes_tree.insert("", "end", values=(recipe.name, recipe.servings)) # Lägg till nya

    def open_new_recipe_popup(self) -> None:
        """
        Skapa popup ruta för att skapa nytt recept
        """

        # Ny ruta
        popup = tk.Toplevel(self.root) # placera överst
        popup.title("Nytt recept")
        popup.geometry("300x200")

        # Attribut
        recipe_name_var = tk.StringVar()
        serving_var = tk.IntVar()

        # Skapa titlar
        ttk.Label(popup, text="Receptnamn:").pack(pady=5)
        ttk.Label(popup, text="Antal portioner:").pack(pady=5)

        # Plocka input
        ttk.Entry(popup, textvariable=recipe_name_var).pack(pady=5)
        ttk.Spinbox(popup, from_=1, to=20, textvariable=serving_var, width=5).pack(pady=5)

        def create_recipe_and_close() -> None:
            """
            Intern metod

            Skapar recept, sparar och stänger popup
            """
            # Hämta attribut
            name = recipe_name_var.get().strip()
            servings = serving_var.get()

            if not name:
                messagebox.showwarning("Fel", "Ange ett namn.")
                return

            # Skapa nytt recept-objekt
            new_recipe = Recipe(name=name, ingredients=[], servings=servings)

            self.recipes.upsert(new_recipe) # Skapar recept
            self.recipes.save() # Spara
            self.refresh_recipes_view()

            messagebox.showinfo("Klart", f"Recept '{name}' skapades!")
            popup.destroy() # ta bort popup

        ttk.Button(popup, text="Skapa", command=create_recipe_and_close).pack(pady=10)

    def open_selected_recipe_ingredients_window(self) -> None:
        """
        Öppna fönster för recepthantering
        """
        selected_rows = self.recipes_tree.selection() # hämta vald

        # Felhantering
        if not selected_rows:
            messagebox.showwarning("Ingen markering", "Välj ett recept att redigera.")
            return

        # Hämta namn
        recipe_name = self.recipes_tree.item(selected_rows[0])["values"][0]

        # Kalla på recept-fönster för specifikt recept
        self.open_recipe_ingredients_window(recipe_name)

    def open_recipe_ingredients_window(self, recipe_name: str) -> None:
        """
        Öppnar recepthanterings-meny för specifikt recept

        :param recipe_name: Namn på recept som ska hanteras
        """

        # Skapa popup
        window = tk.Toplevel(self.root)
        window.title(f"Hantera recept: {recipe_name}")
        window.geometry("420x550")

        # Koppla till fönster
        recipe_mgmt_frame = tk.Frame(window)
        recipe_mgmt_frame.pack(pady=5, fill="x")

        # Titel
        ttk.Label(recipe_mgmt_frame, text="Receptnamn:").grid(row=0, column=0, padx=5)
        recipe_name_var = tk.StringVar(value=recipe_name)

        # Input fält
        recipe_name_entry = ttk.Entry(recipe_mgmt_frame, textvariable=recipe_name_var, width=20)
        recipe_name_entry.grid(row=0, column=1, padx=5)

        def rename_recipe() -> None:
            """
            Intern metod för att byta namn på recept
            """
            new_name = recipe_name_var.get().strip() # Hämta nytt namn

            if not new_name:
                messagebox.showerror("Fel", "Receptnamnet kan inte vara tomt.")
                return

            if new_name != recipe_name:
                if self.recipes.rename(recipe_name, new_name): # Byt namn
                    self.recipes.save() # Spara
                    self.refresh_recipes_view()
                    window.title(f"Hantera recept: {new_name}")
                    messagebox.showinfo("Namnbyte", f"Receptet har bytt namn till '{new_name}'.")

                    window.destroy() # Ta bort fönster
                    self.open_recipe_ingredients_window(new_name)
                else:
                    messagebox.showerror("Fel", f"Kunde inte byta namn på receptet.")

        def delete_recipe() -> None:
            """
            Intern metod för att ta bort specifikt recept
            """
            if messagebox.askyesno("Bekräfta borttagning", f"Är du säker på att du vill ta bort receptet '{recipe_name}'?"):
                if self.recipes.remove(recipe_name):
                    self.recipes.save() # Spara
                    self.refresh_recipes_view()
                    window.destroy() # Stäng fönster
                    messagebox.showinfo("Borttaget", f"Receptet '{recipe_name}' har tagits bort.")
                else:
                    messagebox.showerror("Fel", f"Kunde inte ta bort receptet '{recipe_name}'.")

        # Skapa och koppla knappar
        ttk.Button(recipe_mgmt_frame, text="Byt namn", command=rename_recipe).grid(row=0, column=2, padx=5)
        ttk.Button(recipe_mgmt_frame, text="Ta bort recept", command=delete_recipe).grid(row=0, column=3, padx=5)

        ttk.Separator(window, orient="horizontal").pack(fill="x", pady=5)

        # Skapa yta för skalning
        scaling_frame = tk.Frame(window)
        scaling_frame.pack(pady=5, fill="x", padx=10)

        # Hämta recept
        recipe = self.recipes.get(recipe_name)
        original_servings = recipe.servings if recipe else 1

        # Skapa titlar för skalning av recept
        ttk.Label(scaling_frame, text="Antal portioner:").pack(side="left", padx=5)
        portions_var = tk.IntVar(value=original_servings)
        portions_spinner = ttk.Spinbox(scaling_frame, from_=1, to=20, textvariable=portions_var, width=5) # Input
        portions_spinner.pack(side="left", padx=5)

        # Knapp för att laga recept
        cook_button = ttk.Button(scaling_frame, text="Laga recept", command=lambda: self.cook_recipe_from_window(recipe_name, portions_var.get(), window))
        cook_button.pack(side="right", padx=5)

        ttk.Separator(window, orient="horizontal").pack(fill="x", pady=5)

        # Skapa ingredienser-listan
        ttk.Label(window, text="Ingredienser:").pack(anchor="w", padx=10)
        ingredients_listbox = tk.Listbox(window, width=45, height=12)
        ingredients_listbox.pack(pady=5)

        def refresh_ingredients_list() -> None:
            """
            Uppdaterar ingredienslista vid om-skalning
            """
            ingredients_listbox.delete(0, tk.END)
            recipe = self.recipes.get(recipe_name)
            if recipe:

                scaled_ings = scaled_ingredients(recipe, portions_var.get()) # Skalar recept

                for ingredient in scaled_ings:
                    ingredients_listbox.insert(tk.END, f"{ingredient.name}: {ingredient.quantity} {ingredient.unit}")

        # Uppdaterar ingredienslista vid om-skalning
        portions_var.trace_add("write", lambda *args: refresh_ingredients_list())

        # Lägg till ny yta i popup
        fields_frame = tk.Frame(window)
        fields_frame.pack(pady=5)

        # Skapa titlar
        ttk.Label(fields_frame, text="Namn").grid(row=0, column=0)
        ttk.Label(fields_frame, text="Mängd").grid(row=0, column=1)
        ttk.Label(fields_frame, text="Enhet").grid(row=0, column=2)

        # Hämta imput
        ingredient_name_entry = tk.Entry(fields_frame, width=14)
        ingredient_quantity_entry = tk.Entry(fields_frame, width=8)
        ingredient_unit_combobox = ttk.Combobox(fields_frame, values=self.allowed_units, width=8, state="readonly")

        ingredient_name_entry.grid(row=1, column=0)
        ingredient_quantity_entry.grid(row=1, column=1)
        ingredient_unit_combobox.grid(row=1, column=2)
        ingredient_unit_combobox.set("st")

        def add_ingredient_to_recipe() -> None:
            """
            Intern metod

            Försöker lägga till ingrediens i aktuellt recept
            """

            try:
                name = ingredient_name_entry.get().strip()
                quantity = float(ingredient_quantity_entry.get() or 0)
                unit = ingredient_unit_combobox.get().strip()
                self.recipes.add_ingredient(recipe_name, Ingredient(name, quantity, unit))
                self.recipes.save() # Spara
                refresh_ingredients_list()

            except Exception as error:
                messagebox.showerror("Fel", str(error))

        def remove_selected_ingredient_from_recipe() -> None:
            """
            Intern metod

            Försöker ta bort vald ingrediens i aktuellt recept
            """
            selection = ingredients_listbox.curselection() # Hämta vald
            if not selection:
                return

            name = ingredients_listbox.get(selection[0]).split(":")[0] # Hämta namn
            self.recipes.remove_ingredient(recipe_name, name)
            self.recipes.save() # Spara
            refresh_ingredients_list()

        # Lägg till och koppla knapp
        ttk.Button(window, text="Lägg till ingrediens", command=add_ingredient_to_recipe).pack(pady=2)
        ttk.Button(window, text="Ta bort vald ingrediens", command=remove_selected_ingredient_from_recipe).pack(pady=2)
        ttk.Button(window, text="Stäng", command=window.destroy).pack(pady=5)

        refresh_ingredients_list()

    def cook_recipe_from_window(self, recipe_name: str, portions: int, parent_window=None):
        """
        Lagar recept från popup fönster

        :param recipe_name: Receptets namn
        :param portions: Antal portioner som ska lagas
        :param parent_window: "Undre" fönster, alltid huvudfönstret
        """
        recipe = self.recipes.get(recipe_name)
        if not recipe:
            messagebox.showerror("Fel", f"Receptet '{recipe_name}' hittades inte.")
            return

        try:
            result = self.recipes.make_recipe(
                recipe_name=recipe_name,
                servings=portions,
                inventory=self.inventory,
                shopping=self.shopping
            )

            # Allt fanns
            if result["ok"]:
                self.inventory.save()
                self.refresh_inventory_view()
                messagebox.showinfo(
                    "Klart",
                    f"Recept '{recipe_name}' har lagats med {portions} portioner och inventariet har uppdaterats!"
                )
                if parent_window:
                    parent_window.destroy()
                return

            # Saknade ingredienser
            if result["shopping_list"]:
                self.shopping.save()
                self.refresh_shopping_view()
                messagebox.askyesno(
                    "Varor saknas",
                    f"Du saknar ingredienser för att laga '{recipe_name}'.\n"
                    f"Vill du skapa en inköpslista med skapade ingredienser?"
                )
                return

            # Saknade ingredienser, fråga om att skapa inköpslista
            missing_text = ""
            if result["missing"]:
                if isinstance(result["missing"][0], str):
                    missing_text = "\n".join([f"• {msg}" for msg in result["missing"]])
                else:
                    missing_text = "\n".join([
                        f"• {item.name} ({round(item.quantity, 2)} {item.unit})"
                        for item in result["missing"]
                    ])

            add_to_list = messagebox.askyesno(
                "Varor saknas",
                f"Du saknar följande ingredienser i inventariet för '{recipe_name}':\n\n"
                f"{missing_text}\n\n"
                "Vill du lägga till dessa i inköpslistan?"
            ) # JA/NEJ fönster

            if add_to_list: # Skapa shoppinglista
                shopping_list_name = f"{recipe_name} - inköp"
                self.recipes.make_recipe(
                    recipe_name=recipe_name,
                    servings=portions,
                    inventory=self.inventory,
                    shopping=self.shopping,
                    shopping_list_name=shopping_list_name
                )
                self.shopping.save()
                self.refresh_shopping_view()
                messagebox.showinfo(
                    "Tillagt",
                    f"Ingredienserna har lagts till i inköpslistan '{shopping_list_name}'."
                )
            else:
                messagebox.showinfo("Avbrutet", "Inga ingredienser har lagts till i inköpslistan.")

        except Exception as error:
            messagebox.showerror("Fel", str(error))

    """ Shoppingflik """
    def build_shopping_tab(self):
        container = self.tab_shopping # Koppla tab

        # Skapa kommando-panel
        toolbar = ttk.Frame(container)
        toolbar.pack(fill="x", padx=10, pady=5)

        # Skapa och koppla knappar
        ttk.Button(toolbar, text="Ny lista", command=self.open_new_shopping_list_popup).pack(side="left", padx=5)
        ttk.Button(toolbar, text="Hantera inköpslista", command=self.open_selected_shopping_list_items_window).pack(side="left", padx=5)

        # Skapa layout
        self.shopping_tree = ttk.Treeview(container, columns=("Lista", "Antal varor"), show="headings")
        self.shopping_tree.heading("Lista", text="Lista")
        self.shopping_tree.heading("Antal varor", text="Antal varor")
        self.shopping_tree.pack(fill="both", expand=True, padx=10, pady=10)

        # Bind double-click event to open shopping list management window
        self.shopping_tree.bind("<Double-1>", lambda event: self.open_selected_shopping_list_items_window())

        self.refresh_shopping_view()

    def refresh_shopping_view(self):
        """
        Uppdaterar shopping-fliken
        """
        for row_id in self.shopping_tree.get_children():
            self.shopping_tree.delete(row_id) # Ta bort

        for shopping_list in self.shopping.list_lists():
            self.shopping_tree.insert("", "end", values=(shopping_list.name, len(shopping_list.items))) # Lägg till

    def open_new_shopping_list_popup(self):
        """
        Skapar popup för ny lista
        """
        popup = tk.Toplevel(self.root) # Lägg överst
        popup.title("Ny inköpslista")
        name_var = tk.StringVar() # Koppla till input

        # Skapa inmatningsfält
        ttk.Label(popup, text="Listnamn:").pack(pady=5)
        ttk.Entry(popup, textvariable=name_var).pack(pady=5)

        def create_list_and_close():
            """
            Intern metod

            Skapar en ny lista och sparar den i databasen
            """
            name = name_var.get().strip() # Hämta namn
            if not name:
                return

            empty_list = ShoppingList(name=name, items=[])
            self.shopping.upsert(empty_list) # Lägg till
            self.shopping.save() # Spara
            self.refresh_shopping_view()
            popup.destroy() # Stäng popup

        # Skapa knapp och koppla metod
        ttk.Button(popup, text="Skapa", command=create_list_and_close).pack(pady=10)

    def open_selected_shopping_list_items_window(self):
        """
        Hjälpmetod för att öppna shoppinglista med muspekare
        """
        selected_rows = self.shopping_tree.selection() # Hämta vald rad

        if not selected_rows:
            messagebox.showwarning("Ingen markering", "Välj en inköpslista.")
            return

        # Hämta namn och
        list_name = self.shopping_tree.item(selected_rows[0])["values"][0]
        self.open_shopping_list_items_window(list_name)

    def open_shopping_list_items_window(self, list_name: str):
        """
        Öppnar vald shoppinglista i en popup

        :param list_name: Namn på vald shoppinglista
        """
        # Skapa fönster
        window = tk.Toplevel(self.root)
        window.title(f"Hantera inköpslista: {list_name}")
        window.geometry("420x520")

        # Skapa yta för lista
        list_mgmt_frame = tk.Frame(window)
        list_mgmt_frame.pack(pady=5, fill="x")

        # Skapa titel och koppla input
        ttk.Label(list_mgmt_frame, text="Listnamn:").grid(row=0, column=0, padx=5)
        list_name_var = tk.StringVar(value=list_name)
        list_name_entry = ttk.Entry(list_mgmt_frame, textvariable=list_name_var, width=20)
        list_name_entry.grid(row=0, column=1, padx=5)

        def rename_list():
            """
            Intern metod för att byta namn på shoppinglista
            """
            new_name = list_name_var.get().strip() # Nytt namn

            if not new_name:
                messagebox.showerror("Fel", "Listnamnet kan inte vara tomt.")
                return

            if new_name != list_name:
                if self.shopping.rename(list_name, new_name): # Kalla på metod för namnbyte
                    self.shopping.save() # Spara
                    self.refresh_shopping_view()

                    # Info popup
                    window.title(f"Hantera inköpslista: {new_name}")
                    messagebox.showinfo("Namnbyte", f"Inköpslistan har bytt namn till '{new_name}'.")

                    window.destroy() # Stäng popup
                    self.open_shopping_list_items_window(new_name)
                else:
                    messagebox.showerror("Fel", f"Kunde inte byta namn på inköpslistan.")

        def delete_list():
            """
            Intern metod för att ta bort shoppinglista
            """

            # Fråga om JA/NEJ
            if messagebox.askyesno("Bekräfta borttagning", f"Är du säker på att du vill ta bort inköpslistan '{list_name}'?"):
                if self.shopping.remove(list_name): # Kalla på metod för borttagning
                    self.shopping.save() # Spara
                    self.refresh_shopping_view()
                    window.destroy() # Stäng popup
                    messagebox.showinfo("Borttaget", f"Inköpslistan '{list_name}' har tagits bort.")
                else:
                    messagebox.showerror("Fel", f"Kunde inte ta bort inköpslistan '{list_name}'.")

        def purchase_items():
            """
            Intern metod för att "köpa" varor
            """
            if messagebox.askyesno("Bekräfta köp", f"Vill du lägga till alla varor från '{list_name}' i inventariet?"):
                self.shopping.purchase_item(list_name, self.inventory, clear=True) # Lägg till i inventariet

                # Spara inventarie och shoppinglista
                self.inventory.save()
                self.shopping.save()

                # Uppdatera båda
                self.refresh_inventory_view()
                self.refresh_shopping_view()

                window.destroy() # Stäng popup
                messagebox.showinfo("Köp genomfört", f"Varorna från '{list_name}' har lagts till i inventariet och listan har tömts.")

        # Skapa knappar och koppla till tillhörande metod
        ttk.Button(list_mgmt_frame, text="Byt namn", command=rename_list).grid(row=0, column=2, padx=5) # namnbyte
        ttk.Button(list_mgmt_frame, text="Ta bort lista", command=delete_list).grid(row=0, column=3, padx=5) # borttagning

        ttk.Separator(window, orient="horizontal").pack(fill="x", pady=5)

        # Skapa knapp för köp
        purchase_frame = tk.Frame(window)
        purchase_frame.pack(pady=5, fill="x")
        ttk.Button(purchase_frame, text="Handla alla varor", command=purchase_items).pack(side="right", padx=10)

        ttk.Separator(window, orient="horizontal").pack(fill="x", pady=5)

        # skapa varulista
        ttk.Label(window, text="Varor i listan:").pack(anchor="w", padx=10)
        items_listbox = tk.Listbox(window, width=45, height=12)
        items_listbox.pack(pady=5)

        def refresh_items_list():
            """
            Intern metod

            Uppdaterar innehållet i specifik shoppinglist
            """
            items_listbox.delete(0, tk.END)
            current_list = self.shopping.get(list_name)
            if current_list:
                for item in current_list.items:
                    items_listbox.insert(tk.END, f"{item.name}: {round(item.quantity, 2)} {item.unit}")

        # Skapa interface för att lägg till i inköpslista
        fields_frame = tk.Frame(window)
        fields_frame.pack(pady=5)

        ttk.Label(fields_frame, text="Namn").grid(row=0, column=0)
        ttk.Label(fields_frame, text="Mängd").grid(row=0, column=1)
        ttk.Label(fields_frame, text="Enhet").grid(row=0, column=2)

        item_name_entry = tk.Entry(fields_frame, width=14)
        item_quantity_entry = tk.Entry(fields_frame, width=8)
        item_unit_combobox = ttk.Combobox(fields_frame, values=self.allowed_units, width=8, state="readonly")

        item_name_entry.grid(row=1, column=0)
        item_quantity_entry.grid(row=1, column=1)
        item_unit_combobox.grid(row=1, column=2)
        item_unit_combobox.set("st")

        def add_item_to_list():
            """
            Intern metod

            Lägger till vara i aktuell inköpslista
            """
            try:
                name = item_name_entry.get().strip()
                quantity = float(item_quantity_entry.get() or 0)
                unit = item_unit_combobox.get().strip()
                self.shopping.add_item(list_name, Ingredient(name, quantity, unit)) # Lägg till
                self.shopping.save()
                refresh_items_list()
            except Exception as error:
                messagebox.showerror("Fel", str(error))

        def remove_selected_item_from_list():
            """
            Intern metod

            Tar bort en vara i aktuell inköpslista
            """
            selection = items_listbox.curselection()
            if not selection:
                return
            name = items_listbox.get(selection[0]).split(":")[0]
            self.shopping.remove_item(list_name, name)
            self.shopping.save()
            refresh_items_list()

        # Skapa och koppla knappar
        ttk.Button(window, text="Lägg till vara", command=add_item_to_list).pack(pady=2)
        ttk.Button(window, text="Ta bort vald vara", command=remove_selected_item_from_list).pack(pady=2)
        ttk.Button(window, text="Stäng", command=window.destroy).pack(pady=5)

        refresh_items_list()


    # SQL flik för sql demo

    def build_sql_tab(self):
        container = self.tab_sql

        # Tabeller
        left = ttk.Frame(container)
        left.pack(side="left", fill="y", padx=8, pady=8)

        ttk.Label(left, text="Tabeller").pack(anchor="w")
        self.sql_tables = tk.Listbox(left, height=12)
        self.sql_tables.pack(fill="y", expand=False)
        self.sql_tables.bind("<Double-Button-1>", self._sql_preview_selected_table)

        ttk.Button(left, text="Uppdatera tabeller", command=self._sql_refresh_tables).pack(pady=6)

        # editors och resultat
        right = ttk.Frame(container)
        right.pack(side="left", fill="both", expand=True, padx=8, pady=8)

        ttk.Label(right, text="SQL-kommandon:").pack(anchor="w")
        self.sql_query_text = tk.Text(right, height=6)
        self.sql_query_text.pack(fill="x")

        run_bar = ttk.Frame(right)
        run_bar.pack(fill="x", pady=6)
        ttk.Button(run_bar, text="Kör (SELECT/DDL/DML)", command=self.execute_sql_query).pack(side="left")
        ttk.Button(run_bar, text="Rensa", command=lambda: self.sql_query_text.delete("1.0", "end")).pack(side="left", padx=6)

        # Resultat
        self.sql_result_table = ttk.Treeview(right, show="headings")
        self.sql_result_table.pack(fill="both", expand=True)

        # Status
        self.sql_status = ttk.Label(right, text="Klart.", anchor="w")
        self.sql_status.pack(fill="x", pady=4)

        # Initiera tabeller
        self._sql_refresh_tables()


    def _sql_refresh_tables(self):
        # Läs sqlite-katalog
        rows = self.sql_parser.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
        )
        self.sql_tables.delete(0, tk.END)
        for (name,) in rows:
            self.sql_tables.insert(tk.END, name)


    def _sql_preview_selected_table(self, _event=None):
        sel = self.sql_tables.curselection()
        if not sel: return
        table = self.sql_tables.get(sel[0])
        self._sql_show_select(f"SELECT * FROM {table} LIMIT 200")


    def _sql_show_select(self, query: str):
        # Kör SELECT och visa i tabell
        cur = self.sql_parser.conn.cursor()
        try:
            cur.execute(query)
            cols = [d[0] for d in cur.description] if cur.description else []
            rows = cur.fetchall()
            # Rensa tabell
            self.sql_result_table.delete(*self.sql_result_table.get_children())
            self.sql_result_table["columns"] = cols
            for c in cols:  
                self.sql_result_table.heading(c, text=c)
                self.sql_result_table.column(c, width=120, stretch=True)
            # Insert rows
            for r in rows:
                self.sql_result_table.insert("", "end", values=r)
            self.sql_status.config(text=f"{len(rows)} rader.")
        except Exception as e:
            self.sql_status.config(text=f"Fel: {e}")


    def execute_sql_query(self):
        query = self.sql_query_text.get("1.0", "end").strip()
        if not query:
            return

        # visa resultat i tabell
        first = query.lstrip().split(None, 1)[0].lower() if query.strip() else ""
        if first == "select":
            self._sql_show_select(query)
            return

        try:
            # skript för flera kommandon
            self.sql_parser.conn.executescript(query)
            self.sql_parser.conn.commit()
            self.sql_status.config(text="OK (skrivande kommandon körda).")
            # Uppdatera tabellista
            self._sql_refresh_tables()
            # Rensa tabell
            self.sql_result_table.delete(*self.sql_result_table.get_children())
            self.sql_result_table["columns"] = ()
        except Exception as e:
            self.sql_status.config(text=f"Fel: {e}")


In [29]:
root = tk.Tk()
app = App(root, inventory, recipes, shopping, sql_parser)
root.mainloop()
